In [8]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import os
import requests
import time
import json
import warnings
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (mean_squared_error, r2_score,
                             accuracy_score, classification_report,
                             confusion_matrix)
from sklearn.impute import SimpleImputer
warnings.filterwarnings("ignore")

print("✅ All imports successful")

✅ All imports successful


In [9]:
url = "https://SDMDataAccess.sc.egov.usda.gov/Tabular/post.rest"

def query_soil(sql, label=""):
    response = requests.post(
        url,
        data={"query": sql, "format": "JSON+COLUMNNAME"},
        timeout=30
    )
    if response.status_code == 200:
        data = response.json()
        # Convert to DataFrame
        cols = data["Table"][0]
        rows = data["Table"][1:]
        df = pd.DataFrame(rows, columns=cols)
        print(f"✅ {label} — {len(df)} rows returned")
        return df
    else:
        print(f"❌ Error {response.status_code}:", response.text)
        return None

In [ ]:
# https://sdmdataaccess.sc.egov.usda.gov/documents/TableColumnDescriptionsReport.pdf

texas_features = query_soil("""
    SELECT 
        -- 1. Basic Information
        l.areasymbol, 
        m.musym, 
        m.muname,
        c.compname,
        s.saverest,
        
        -- 2. Soil Features (Category/Attribute)
        c.taxorder,        -- Soil Type (Order)
        c.drainagecl,      -- Drainage Class (e.g., Well drained)
        c.elev_r,          -- Elevation (Representative)
        c.slope_r,         -- Slope (Representative)
        
        -- 3. Soil Horizon Numerical Data (from chorizon table)
        ch.hzdept_r,       -- Top depth of horizon
        ch.hzdepb_r,       -- Bottom depth of horizon (Depth to restrictive layer)
        ch.ph1to1h2o_r,    -- pH (1:1 Water)
        ch.om_r,           -- Organic Matter (Proxy for N-P-K)
        ch.ec_r,           -- Electrical Conductivity (Salinity)
        ch.cec7_r,         -- Cation Exchange Capacity (Nutrient storage)
        ch.awc_r,          -- Available Water Capacity (Water storage)
                                
        -- 4. Crop data
        cp.cropname,       -- Crop type (e.g., Corn, Soybeans, Cotton)
        cp.yldunits,       -- Crop yield units per area
        cp.nonirryield_r,   -- The expected yield per acre without supplemental irrigation.
        cp.irryield_r      -- The expected yield per acre with irrigation.
                      

    FROM legend l
    INNER JOIN sacatalog s ON s.areasymbol = l.areasymbol
    INNER JOIN mapunit m ON m.lkey = l.lkey
    INNER JOIN component c ON c.mukey = m.mukey
    INNER JOIN chorizon ch ON ch.cokey = c.cokey
    LEFT OUTER JOIN cocropyld cp ON cp.cokey = c.cokey

    WHERE l.areasymbol LIKE 'TX%'
    AND c.majcompflag = 'Yes'
    AND ch.hzdept_r = 0 -- Wanna see the soil where the root is in
""","texas")


# ─────────────────────────────────────────
# 4. Save to CSV
# ─────────────────────────────────────────
for df, name in [
    (texas_features, "texas"),
]:
    if df is not None:
        df.to_csv(f"{name}.csv", index=False)
        print(f"💾 Saved {name}.csv")

❌ Error 400: <?xml version='1.0' encoding="UTF-8" standalone="no" ?>
<ServiceExceptionReport xmlns="http://www.opengis.net/ogc" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.opengis.net/ogc http://schemas.opengis.net/wms/1.1.1/OGC-exception.xsd">
<ServiceException>
Maximum allowable number of returned records (100000) exceeded.</ServiceException>
</ServiceExceptionReport>



In [2]:
NOAA_TOKEN = "ehfXSNJNJxddiDVkaZTwEOCgMWGrjQNB"
YEARS      = list(range(2010, 2024))
STATES     = {"Texas": "FIPS:48"}
 
# plt.rcParams.update({
#     "figure.facecolor": "#F7F9FC", "axes.facecolor": "#F7F9FC",
#     "axes.spines.top": False, "axes.spines.right": False,
# })
# C = {"indiana":"#2E86AB","iowa":"#A23B72","corn":"#F18F01",
#      "teal":"#44BBA4","red":"#E94F37","purple":"#7F77DD"}
 
# print("=" * 65)
# print("  APPROACH 2: SOIL + WEATHER + YIELD PIPELINE")
# print("=" * 65)

In [4]:
TARGET_TYPES = ["PRCP", "TMAX", "TMIN", "SRAD", "TSUN"]

def fetch_full_year_monthly_stats(years, token):
    all_data_frames = []
    url = "https://www.ncdc.noaa.gov/cdo-web/api/v2/data"
    headers = {"token": token}

    for year in years:
        periods = [("-01-01", "-06-30"), ("-07-01", "-12-31")]
        
        for start_suffix, end_suffix in periods:
            print(f"📡 Fetching {year}{start_suffix} to {end_suffix}...")
            params = {
                "datasetid": "GHCND",
                "locationid": "FIPS:48", # Texas
                "startdate": f"{year}{start_suffix}",
                "enddate": f"{year}{end_suffix}",
                "limit": 1000,
                "units": "metric",
                "datatypeid": TARGET_TYPES
            }

            try:
                r = requests.get(url, headers=headers, params=params, timeout=30)
                if r.status_code == 200:
                    res = r.json().get("results", [])
                    if res:
                        all_data_frames.append(pd.DataFrame(res))
                time.sleep(0.5) 
            except Exception as e:
                print(f"  ❌ Failed: {e}")

    if not all_data_frames:
        return None
        
    full_df = pd.concat(all_data_frames)
    full_df['date'] = pd.to_datetime(full_df['date'])
    full_df['month'] = full_df['date'].dt.month
    
    all_months = range(1, 13)

    precip = (full_df[full_df['datatype'] == 'PRCP']
              .groupby('month')['value'].mean() / 10.0 * 30).reindex(all_months)
    
    tmax = (full_df[full_df['datatype'] == 'TMAX']
            .groupby('month')['value'].mean() / 10.0).reindex(all_months)
    
    tmin = (full_df[full_df['datatype'] == 'TMIN']
            .groupby('month')['value'].mean() / 10.0).reindex(all_months)
    
    if 'SRAD' in full_df['datatype'].values:
        srad = (full_df[full_df['datatype'] == 'SRAD']
                .groupby('month')['value'].mean()).reindex(all_months)
    else:
        srad = pd.Series([np.nan] * 12, index=all_months)

    monthly_stats = pd.DataFrame({
        'month': all_months,
        'avg_precip_mm': precip.values,
        'avg_tmax_c': tmax.values,
        'avg_tmin_c': tmin.values,
        'avg_srad': srad.values
    })

    monthly_stats.to_csv("texas_annual_monthly_stats_2014_2024.csv", index=False)
    return monthly_stats

YEARS = range(2014, 2025)
df_full_year = fetch_full_year_monthly_stats(YEARS, NOAA_TOKEN)

📡 Fetching 2014-01-01 to -06-30...
📡 Fetching 2014-07-01 to -12-31...
📡 Fetching 2015-01-01 to -06-30...
📡 Fetching 2015-07-01 to -12-31...
📡 Fetching 2016-01-01 to -06-30...
📡 Fetching 2016-07-01 to -12-31...
📡 Fetching 2017-01-01 to -06-30...
📡 Fetching 2017-07-01 to -12-31...
📡 Fetching 2018-01-01 to -06-30...
📡 Fetching 2018-07-01 to -12-31...
📡 Fetching 2019-01-01 to -06-30...
📡 Fetching 2019-07-01 to -12-31...
📡 Fetching 2020-01-01 to -06-30...
📡 Fetching 2020-07-01 to -12-31...
📡 Fetching 2021-01-01 to -06-30...
📡 Fetching 2021-07-01 to -12-31...
📡 Fetching 2022-01-01 to -06-30...
📡 Fetching 2022-07-01 to -12-31...
📡 Fetching 2023-01-01 to -06-30...
📡 Fetching 2023-07-01 to -12-31...
📡 Fetching 2024-01-01 to -06-30...
📡 Fetching 2024-07-01 to -12-31...


In [5]:
check_df = pd.read_csv("texas_weather_5items.csv")
print(check_df.head())

print("\n--- Missing Values Check ---")
print(check_df.isnull().sum())

   year  state  precip_mm    tmax_c    tmin_c  srad_avg  sunshine_total_min
0  2014  Texas       1.78  2.802274  1.162609       NaN                 NaN
1  2015  Texas      15.16  2.785936  1.489788       NaN                 NaN
2  2016  Texas      61.75  2.475979  0.967885       NaN                 NaN
3  2017  Texas      38.15  2.786093  1.090647       NaN                 NaN
4  2018  Texas       4.53  2.667568  1.069586       NaN                 NaN

--- Missing Values Check ---
year                   0
state                  0
precip_mm              0
tmax_c                 0
tmin_c                 0
srad_avg              10
sunshine_total_min    10
dtype: int64


In [ ]:
# 1. Download/Load the Station List to identify Texas locations
# This file contains metadata for all stations
stations_url = "https://www.ncei.noaa.gov/pub/data/ghcn/daily/ghcnd-stations.txt"

# Stations file is fixed-width format (FWF)
# Column 31-33 contains the State code
stations_df = pd.read_fwf(stations_url, header=None, 
                          widths=[11, 9, 10, 7, 3, 31, 3], 
                          names=['station', 'lat', 'lon', 'elev', 'state', 'name', 'gsn'])

# Get list of Station IDs located in Texas (TX)
texas_station_ids = stations_df[stations_df['state'] == 'TX']['station'].unique()

print(f"✅ Found {len(texas_station_ids)} stations in Texas.")

# 2. Process the 2025 massive CSV with the TX filter
url = "https://www.ncei.noaa.gov/pub/data/ghcn/daily/by_year/2025.csv.gz"
cols = ['station', 'date', 'element', 'value', 'm_flag', 'q_flag', 's_flag', 'obs_time']
target_elements = ['PRCP', 'TMAX', 'TMIN']

df_chunk = pd.read_csv(url, compression='gzip', names=cols, chunksize=200000, low_memory=False)

texas_data_list = []

print("⏳ Filtering 2025 data for Texas stations only...")

for chunk in df_chunk:
    # Filter by BOTH: 1. Element Type AND 2. Texas Station IDs
    is_target_element = chunk['element'].isin(target_elements)
    is_texas_station = chunk['station'].isin(texas_station_ids)
    
    target = chunk[is_target_element & is_texas_station].copy()
    texas_data_list.append(target)

# Final Concatenation
Full_texas_2025 = pd.concat(texas_data_list)

print(f"✅ Finished! Final record count for Texas: {len(Full_texas_2025)}")

✅ Found 6472 stations in Texas.
⏳ Filtering 2025 data for Texas stations only...
✅ Finished! Final record count for Texas: 923360


In [ ]:
# 1. Format date and extract month
# The 'date' column in NOAA is YYYYMMDD as integer, so we convert it
Full_texas_2025['date'] = pd.to_datetime(Full_texas_2025['date'], format='%Y%m%d')
Full_texas_2025['month'] = Full_texas_2025['date'].dt.month

# 2. Pivot to group by month and element
# We calculate the mean value for each month
monthly_pivot = Full_texas_2025.pivot_table(
    index='month', 
    columns='element', 
    values='value', 
    aggfunc='mean'
)

# 3. Unit Adjustment and Scaling
weather_summary_2025 = pd.DataFrame(index=range(1, 13))
weather_summary_2025['precip_total_mm'] = (monthly_pivot['PRCP'] / 10.0 * 30).reindex(range(1, 13))
weather_summary_2025['tmax_avg_c'] = (monthly_pivot['TMAX'] / 10.0).reindex(range(1, 13))
weather_summary_2025['tmin_avg_c'] = (monthly_pivot['TMIN'] / 10.0).reindex(range(1, 13))

# 3.5 Add 'month' column while keeping the index
# This copies the index values into a new column named 'month'
weather_summary_2025['month'] = weather_summary_2025.index

# Reorder columns to put 'month' at the beginning (Optional but cleaner)
cols = ['month', 'precip_total_mm', 'tmax_avg_c', 'tmin_avg_c']
weather_summary_2025 = weather_summary_2025[cols]

# 4. Export to CSV
weather_summary_2025.to_csv("texas_weather_2025.csv", index=False)

print("--- Monthly Weather Summary 2024 with Month Column ---")
print(weather_summary_2025)

--- Monthly Weather Summary 2024 with Month Column ---
    month  precip_total_mm  tmax_avg_c  tmin_avg_c
1       1        68.904092   12.620515    0.145149
2       2        41.617366   18.198243    4.128925
3       3        60.824599   25.318835    9.548430
4       4        83.453419   27.084042   14.263100
5       5       166.870902   29.932103   17.432293
6       6       132.927817   33.106932   21.867188
7       7       120.738690   33.730676   22.498972
8       8        66.663554   35.028900   22.299706
9       9        56.636780   32.621168   19.164194
10     10        55.588615   29.634530   15.044252
11     11        54.964388   24.378544    9.938428
12     12        14.875720   19.257336    5.518915
